# Lab 03: Parquet Internals, Silver Cleaning Part 1 & Schema Enforcement
**Course:** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Date & Format:** Lab #3 (EN: Mon 23.11.2026 afternoon *following Tutorial #2* / FR: Tue 24.11.2026) · 3 Hours (180 min)

---

## 🎯 1. Context & Architecture-as-Code Lineage
In Lab 02 you staged raw data verbatim into `bronze_staged/*.parquet`. Before performing statistical outlier math (Lab 04), today's lab connects **Lecture 02's Apache Parquet internals** directly to **Silver Structural Hygiene & Strict Schema Typing**, aligned with **DAMA Data Quality Rules (`DQR`)** from Tutorial #2:

```mermaid
flowchart TD
    B0["2. Bronze Stage: bronze_staged/*.parquet"] --> P0["3. Parquet Internals & Pushdown (3.1 - 3.4)"]
    P0 --> Q0["4. Read-Only Bronze DQ Profiling & Scorecard (4.1 - 4.3)"]
    Q0 --> S1["5. String Hygiene & Canonical Key Aliasing"]
    S1 --> S2["6. Data Types, Timestamp Parsing, Keep-Last Dedup & S3 Rollover (6.1 - 6.2)"]
    S2 --> P1["7. Enforce Silver Schema Contract -> silver_part1/*.parquet"]
```

**Your Objectives:**
1. **Inspect Apache Parquet Internals (Lecture 02 Bridge):** Verify `PAR1` magic header/footer bytes, decode the footer size, inspect Row Groups, Column Chunks, byte offsets, `RLE_DICTIONARY`/`RLE`/`SNAPPY` compression, footer min/max/null statistics, and **Projection + Predicate Pushdown**.
2. **Profile Bronze (Read-Only):** Compile a **Bronze DAMA DQ Scorecard** without mutating Bronze.
3. **String Hygiene & Aliases:** Resolve whitespace, inconsistent casing (`'Press_01 '`), legacy aliases (`'P-02'` -> `'PRESS_02'`), and mixed CMMS `downtime_type` strings (`'unplanned'` -> `'Unplanned'`).
4. **Understand & Enforce Silver Data Types:** Prove why storing timestamps/dates as strings breaks arithmetic, ordering, RAM, and SQL pushdown; parse timestamps (`timestamp[ns]`, `timestamp[ns, tz=UTC]`) and dates (`date32[day]`), deduplicate (`keep='last'`), and fix **Night Shift (`S3`: `22:00–06:00`) midnight crossovers** via `(timestamp - 6h)`.
5. **Persist & Audit Schema Contracts:** Save all 5 typed tables to `silver_part1/*.parquet` and verify embedded Arrow/Parquet schemas and DuckDB `DESCRIBE` types.

---

## ⚙️ 2. Environment Setup & Bronze Input Discovery

### Step 1: Resolve Bronze Inputs, PyArrow/DuckDB Tooling & Silver Part 1 Output Directory

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case1_smartpress_automotive"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

BRONZE_STAGED = resolve_stage_dir("bronze_staged")
SILVER_P1 = SANDBOX_DATA / "silver_part1"
SILVER_P1.mkdir(parents=True, exist_ok=True)

df_telem = pd.read_parquet(BRONZE_STAGED / "bronze_telemetry.parquet")
df_cmms = pd.read_parquet(BRONZE_STAGED / "bronze_maintenance.parquet")
df_machines = pd.read_parquet(BRONZE_STAGED / "bronze_machines.parquet")
df_shifts = pd.read_parquet(BRONZE_STAGED / "bronze_shifts.parquet")
df_variants = pd.read_parquet(BRONZE_STAGED / "bronze_product_variants.parquet")

print("Bronze source :", BRONZE_STAGED)
print("Silver Part 1 :", SILVER_P1)
print(f"Loaded telemetry: {len(df_telem):,} rows | CMMS tickets: {len(df_cmms):,} rows")


## 📦 3. Inside Apache Parquet — Anatomy, Header, Footer & Pushdown (Lecture 02 Bridge)
In **Lecture 02 (§5.3)** you saw why Apache Parquet beats CSV/JSON for analytical storage. Let's open `bronze_telemetry.parquet` as raw binary bytes and via `pyarrow.parquet.ParquetFile`—**without loading the data rows**—to verify every part of its internal layout:

```text
 [0 .. 4)              Magic Header: b'PAR1' (4 bytes)
 [4 .. footer_start)   Row Groups -> Column Chunks -> Pages (Dictionary + RLE + Snappy)
 [footer_start .. -8)  File Footer Metadata (Thrift: schema, byte offsets, min/max/null stats)
 [-8 .. -4)            4-byte little-endian integer = Footer length in bytes
 [-4 .. EOF)           Magic Footer: b'PAR1' (4 bytes)
```

### 3.1 Demo: Magic Bytes (`PAR1`), Footer Length, Metadata & Embedded Schema
**Why is the File Footer placed at the end and read FIRST?** Because while writing, the engine does not know the final byte offsets or min/max statistics until the last row is flushed. When reading, Parquet jumps directly to `EOF - 8`, reads the 4-byte `footer_len`, and loads the File Footer metadata before touching a single data page!

*Run the ready-to-launch demo cell below to inspect the raw bytes and embedded Arrow schema of `bronze_telemetry.parquet`.*

In [ ]:
# Ready-to-run Demo: Inspect the raw binary structure and metadata footer of a Parquet file
pq_path = BRONZE_STAGED / "bronze_telemetry.parquet"

# Read the entire file as raw bytes so we can verify its binary header/footer layout
raw_bytes = pq_path.read_bytes()

# 1. Verify the 4-byte "PAR1" magic signature at the very start and very end of the file
magic_header = raw_bytes[:4]
magic_footer = raw_bytes[-4:]

# 2. The 4 bytes right before the final "PAR1" (indices -8..-4) store the Footer length
#    as a little-endian integer (least-significant byte first, per the Parquet spec)
footer_len = int.from_bytes(raw_bytes[-8:-4], "little")

# 3. Open ONLY the Parquet metadata handle — this reads the Footer, NOT the row data pages
pf = pq.ParquetFile(pq_path)

# Sanity checks: confirm valid PAR1 markers and that our decoded footer_len matches PyArrow
assert magic_header == b"PAR1" and magic_footer == b"PAR1", "Invalid Parquet magic bytes!"
assert footer_len == pf.metadata.serialized_size, "Footer length mismatch!"

print(f"File size         : {len(raw_bytes):,} bytes")
print(f"Magic header/tail : {magic_header} ... {magic_footer}")
print(f"Footer length     : {footer_len:,} bytes (matches pf.metadata.serialized_size)")
print(f"Rows / Cols / RGs : {pf.metadata.num_rows:,} rows | {pf.metadata.num_columns} cols | {pf.metadata.num_row_groups} Row Group(s)")
print(f"Format / Writer   : v{pf.metadata.format_version} | {pf.metadata.created_by}")
print(f"Custom KV keys    : {list((pf.metadata.metadata or {}).keys())}")
print("\nEmbedded Arrow Schema (notice 'timestamp' & '_ingestion_timestamp_utc' are still STRING in Bronze!):")
print(pf.schema_arrow)


### 3.2 Demo: Column Chunks, Byte Offsets, Encodings (`Dictionary` + `RLE` + `Snappy`) & Footer Statistics
Run this demo to inspect `pf.metadata.row_group(0)` column-by-column **strictly from the metadata footer** (without reading or decompressing any data pages) and observe:
1. **Columnar Byte Layout:** Each Column Chunk starts right after the previous chunk (`dictionary_page_offset` / `data_page_offset`), and `4 + sum(compressed_B) + footer_len + 8` equals the **exact byte size of the file**!
2. **Dictionary + RLE Compression:** Look at `_source_file` and `_ingestion_timestamp_utc`: `23,164` repeated strings compress down to ~`200–265 bytes`!
3. **Free DQ Diagnostics from Footer Statistics:** Look at `nulls`, `min`, and `max`—we can already spot `120` missing temperatures, `-999.0`/`4850.0 Nm` torque, `-50` units, `'P-02'`..`'press_03'`, and `'2099-01-01'` *without decompressing a single row*!

In [ ]:
# Ready-to-run Demo: Read per-column layout & min/max/null stats directly from Footer metadata
rg = pf.metadata.row_group(0)

chunk_rows = []
for i in range(rg.num_columns):
    # c = metadata descriptor for column i; st = precomputed min/max/null_count stats
    c = rg.column(i)
    st = c.statistics
    chunk_rows.append({
        "column": c.path_in_schema,                 # Column name in the file schema
        "phys_type": c.physical_type,               # Low-level Parquet storage type (INT64, DOUBLE, BYTE_ARRAY)
        "comp": c.compression,                      # Compression codec (SNAPPY)
        "encodings": ",".join(c.encodings),         # Encodings used (Dictionary + RLE + PLAIN)
        "dict_off": c.dictionary_page_offset,       # Byte offset where the dictionary page starts
        "data_off": c.data_page_offset,             # Byte offset where the compressed data pages start
        "raw_B": c.total_uncompressed_size,         # Uncompressed size of this column chunk (bytes)
        "comp_B": c.total_compressed_size,          # Compressed size on disk (bytes)
        "nulls": st.null_count if st else None,     # Missing values count stored in footer
        "min_stat": repr(st.min) if st and st.has_min_max else None,  # Min value in Row Group
        "max_stat": repr(st.max) if st and st.has_min_max else None,  # Max value in Row Group
    })

df_chunks = pd.DataFrame(chunk_rows)

# Verify the exact binary byte budget:
# 4B header ("PAR1") + all compressed column chunks + Thrift footer + 8B tail (4B len + 4B "PAR1")
total_payload_b = int(df_chunks["comp_B"].sum())
assert 4 + total_payload_b + footer_len + 8 == len(raw_bytes), "Byte map mismatch!"

print(f"Exact binary map: 4B (PAR1) + {total_payload_b:,}B (Column Chunks) + {footer_len:,}B (Footer) + 8B (len+PAR1) = {len(raw_bytes):,}B")
print("\nColumn Chunk Metadata & Footer Statistics (Row Group 0):")
print(df_chunks.to_string(index=False))


### 3.3 Demo: Projection Pushdown (`columns=[...]`)
Reading only `["machine_id", "torque_Nm"]` uses the byte offsets in the Parquet footer to jump directly to those 2 columns and skip >80% of Parquet column chunk bytes (and >96% compared to scanning the raw CSV!).

In [ ]:
# Ready-to-run Demo: Projection Pushdown (skip unused columns using Footer byte offsets)
selected_cols = ["machine_id", "torque_Nm"]
csv_bytes = (BRONZE_RAW / "sensor_telemetry.csv").stat().st_size if (BRONZE_RAW / "sensor_telemetry.csv").exists() else 0

# PyArrow reads ONLY the 2 requested columns instead of all 12
tbl_proj = pq.read_table(pq_path, columns=selected_cols)

# Sum the compressed byte sizes of only those 2 Column Chunks from Row Group 0 metadata
proj_bytes = sum(
    rg.column(i).total_compressed_size
    for i in range(rg.num_columns)
    if rg.column(i).path_in_schema in selected_cols
)
print(f"Projection Pushdown ({selected_cols}):")
print(f"  Raw CSV scan: {csv_bytes:,} B -> Full Parquet chunks: {total_payload_b:,} B -> Projected 2 cols: {proj_bytes:,} B ({100 * (1 - proj_bytes / total_payload_b):.1f}% Parquet I/O skipped!)")
print(f"  Projected table shape: {tbl_proj.num_rows:,} rows x {tbl_proj.num_columns} cols")


### 3.4 Demo: Multi-Row-Group Predicate Pushdown (`filters=[...]`)
When a Parquet file spans multiple Row Groups (`row_group_size=6000`), the query engine inspects each Row Group's footer `[min, max]` and **skips entire Row Groups** whose range cannot match! Notice also how uncleaned `'press_03'` lands in Row Group 3 (`'p' > 'P'` in ASCII), proving why Section 5 string hygiene matters for filtering.

In [ ]:
# Ready-to-run Demo: Multi-Row-Group Predicate Pushdown (skip whole Row Groups using [min, max] stats)
# Write an in-memory Parquet buffer sorted by machine_id and split into 4 Row Groups
sink = pa.BufferOutputStream()
pq.write_table(
    pa.Table.from_pandas(df_telem.sort_values("machine_id"), preserve_index=False),
    sink,
    row_group_size=6000,   # Splits 23,164 rows into 4 Row Groups (~6,000 rows each)
    compression="snappy",
)
multi_buf = sink.getvalue()
pf_multi = pq.ParquetFile(pa.BufferReader(multi_buf))

# Apply a filter pushdown: PyArrow checks each Row Group's [min, max] footer stats BEFORE reading data
tbl_pred = pq.read_table(
    pa.BufferReader(multi_buf),
    columns=["timestamp", "machine_id", "torque_Nm"],
    filters=[("machine_id", "=", "PRESS_03")],
)

print(f"Multi-Row-Group Stats ({pf_multi.metadata.num_row_groups} Row Groups, sorted by machine_id):")
for r in range(pf_multi.metadata.num_row_groups):
    rg_r = pf_multi.metadata.row_group(r)
    st_m = rg_r.column(1).statistics  # column 1 is machine_id
    # If "PRESS_03" falls outside [min, max], the entire Row Group is skipped without decompression
    can_skip = not (st_m.min <= "PRESS_03" <= st_m.max)
    print(f"  RG {r}: rows={rg_r.num_rows:>4} | machine_id=[{st_m.min!r}, {st_m.max!r}] -> {'SKIPPED by footer!' if can_skip else 'CANDIDATE'}")
print(f"Predicate Pushdown result ('PRESS_03'): {tbl_pred.num_rows:,} rows (notice uncleaned 'press_03' in RG 3 was missed!)")


## 🔍 4. Initial Bronze Data Quality Profiling (Strictly Read-Only!)
**Critical Rule (`Bronze Immutability`):** Now that the Parquet footer already hinted at dirty keys, `120` nulls, extreme torques, and `2099-*` dates, play forensic detective on the staged Bronze tables (`df_telem`, `df_cmms`, `df_machines`) so our **DAMA DQ Rules** catch 100% of anomalies.

### 4.1 DQ Scan: Machine Key & Category Casing Drift (`machine_id` & `downtime_type`)
Compare raw keys against the canonical ERP master list (`PRESS_01..PRESS_04`).

In [ ]:
# Canonical ERP machine set (e.g. {'PRESS_01', 'PRESS_02', 'PRESS_03', 'PRESS_04'})
valid_machines = set(df_machines["machine_id"].astype(str).str.strip().str.upper())

# TODO: Inspect raw telemetry `machine_id` values using repr() (so trailing spaces show!),
# and CMMS `downtime_type` value_counts() to spot casing drift.
raw_machine_counts = None     # REPLACE WITH: df_telem["machine_id"].value_counts(dropna=False)
cmms_type_counts = None       # REPLACE WITH: df_cmms["downtime_type"].value_counts(dropna=False)

if raw_machine_counts is not None and cmms_type_counts is not None:
    print("Canonical ERP machines:", sorted(valid_machines))
    print("\n1. Raw SCADA machine_id strings (notice 'Press_01 ', 'P-02', 'press_03' & orphan 'PRESS_99'):")
    for k, cnt in raw_machine_counts.items():
        print(f"  {repr(k):<14} -> {cnt:>6,} rows")
    print("\n2. CMMS downtime_type casing drift ('Unplanned' vs 'unplanned' vs 'Preventive'):")
    print(cmms_type_counts.to_string())
else:
    print("⏳ [TODO] Assign raw_machine_counts and cmms_type_counts above.")


### 4.2 DQ Scan: Physics Bounds, Nulls (`NaN`), Future Dates & Duplicates
Apply vectorized boolean masks (`<`, `>`, `|`, `~`, `.isin()`) to diagnose:
1. Impossible mechanical shaft torque (`< 0` like `-999.0 Nm` or `> 200.0` like `4850.0 Nm`).
2. Missing thermocouple readings (`NaN` in `process_temp_K`) — remember: **never `dropna()`**, or you delete valid production units!
3. `scrap_units > produced_units`, future `2099-*` timestamps, duplicate telemetry rows, and orphan/future CMMS tickets (`CMMS-20269999`).

In [ ]:
# TODO: Complete each boolean/numeric diagnostic counter below using vectorized Pandas masks
bad_torque_count = None   # REPLACE WITH: ((df_telem["torque_Nm"] < 0) | (df_telem["torque_Nm"] > 200)).sum()
null_temp_count = None    # REPLACE WITH: df_telem["process_temp_K"].isna().sum()
dupes_count = None        # REPLACE WITH: df_telem.duplicated(subset=["timestamp", "machine_id", "product_variant"]).sum()

tel_ts = pd.to_datetime(df_telem["timestamp"], errors="coerce")
future_tel_count = (tel_ts > "2027-01-01").sum()
bad_scrap_count = (df_telem["scrap_units"] > df_telem["produced_units"]).sum()

# CMMS checks
mnt_ts = pd.to_datetime(df_cmms["event_date"], errors="coerce")
mnt_bad_mask = (~df_cmms["machine_code"].astype(str).str.strip().str.upper().isin(valid_machines)) | mnt_ts.isna() | (mnt_ts > "2027-01-01")

if all(v is not None for v in (bad_torque_count, null_temp_count, dupes_count)):
    print(f"Telemetry out-of-bounds torque (<0 or >200 Nm): {bad_torque_count}")
    print(f"Telemetry missing process_temp_K (NaN)      : {null_temp_count}  (to interpolate in Lab 04, NOT drop!)")
    print(f"Telemetry MQTT duplicate rows               : {dupes_count}")
    print(f"Telemetry future 2099 timestamps            : {future_tel_count}")
    print(f"Telemetry scrap_units > produced_units      : {bad_scrap_count}")
    print("\nFlagged CMMS rows (non-canonical alias, orphan press, or 2099-* timestamp):")
    display(df_cmms.loc[mnt_bad_mask, ["ticket_id", "event_date", "machine_code", "downtime_type", "failure_mode", "repair_duration_min"]].head(8))
    print("  Total CMMS rows flagged above:", mnt_bad_mask.sum(), "| Defensive repair_duration_min <= 0:", (df_cmms["repair_duration_min"] <= 0).sum())
else:
    print("⏳ [TODO] Complete bad_torque_count, null_temp_count, and dupes_count above.")


### 4.3 Bronze DAMA DQ Scorecard (Triage Plan for Labs 03–04)
Compile your read-only findings into a **`DataFrame` Scorecard** mapping every detected anomaly to its **Lab 03 (Normalize/Deduplicate)** or **Lab 04 (Interpolate/Quarantine)** action.

In [ ]:
if all(v is not None for v in (bad_torque_count, null_temp_count, dupes_count)):
    orphan_tel_count = (df_telem["machine_id"].astype(str).str.strip().str.upper() == "PRESS_99").sum()
    alias_tel_count = df_telem["machine_id"].isin(["Press_01 ", "P-02", "press_03"]).sum()
    cmms_future_orphan_count = (mnt_ts > "2027-01-01").sum()
    cmms_unplanned_lower_count = (df_cmms["downtime_type"] == "unplanned").sum()

    dq_scorecard = pd.DataFrame([
        {"Rule_ID": "DQR-KEY", "Source": "SCADA + CMMS", "Anomaly": "Whitespace/casing/alias drift ('Press_01 ', 'P-02', 'unplanned')", "Count": alias_tel_count + cmms_unplanned_lower_count, "Planned_Action": "Lab 03 Normalize (.strip().upper() + alias map)"},
        {"Rule_ID": "DQR-107", "Source": "SCADA CSV", "Anomaly": "MQTT duplicate telemetry rows", "Count": dupes_count, "Planned_Action": "Lab 03 Deduplicate (keep='last')"},
        {"Rule_ID": "DQR-101", "Source": "SCADA + CMMS", "Anomaly": "Orphan ghost asset (PRESS_99)", "Count": orphan_tel_count, "Planned_Action": "Lab 04 Quarantine (tier BLOCKER)"},
        {"Rule_ID": "DQR-102", "Source": "SCADA + CMMS", "Anomaly": "Future 2099-* timestamps (incl. CMMS-20269999)", "Count": future_tel_count + cmms_future_orphan_count, "Planned_Action": "Lab 04 Quarantine"},
        {"Rule_ID": "DQR-103", "Source": "SCADA CSV", "Anomaly": "Impossible torque_Nm (-999.0 / 4850.0 Nm)", "Count": bad_torque_count, "Planned_Action": "Lab 04 Quarantine"},
        {"Rule_ID": "DQR-104", "Source": "SCADA CSV", "Anomaly": "scrap_units > produced_units", "Count": bad_scrap_count, "Planned_Action": "Lab 04 Quarantine"},
        {"Rule_ID": "DQR-TEMP", "Source": "SCADA CSV", "Anomaly": "Missing process_temp_K (NaN)", "Count": null_temp_count, "Planned_Action": "Lab 04 Time Interpolate (NO dropna!)"},
    ])
    print("=== BRONZE DATA QUALITY SCORECARD (TRIAGE PLAN) ===")
    display(dq_scorecard)
else:
    print("⏳ Complete the DQ Scan cells above to generate your Bronze DQ Scorecard.")


## 🧹 5. String Hygiene & Canonical Key Aliasing
Normalize `machine_id`, `machine_code`, and mixed `downtime_type` labels deterministically before deduplication.

In [ ]:
# Canonical alias lookup table mapping legacy variants -> standard ERP machine IDs
MACHINE_ALIASES = {
    "P-01": "PRESS_01", "PRESS-01": "PRESS_01", "P_01": "PRESS_01",
    "P-02": "PRESS_02", "PRESS-02": "PRESS_02", "P_02": "PRESS_02",
    "P-03": "PRESS_03", "PRESS-03": "PRESS_03", "P_03": "PRESS_03",
    "P-04": "PRESS_04", "PRESS-04": "PRESS_04", "P_04": "PRESS_04",
}

def clean_machine_code(val):
    if pd.isna(val):                              # Guard against missing/NaN machine codes
        return "UNKNOWN"
    s = str(val).strip().upper()                  # Remove trailing spaces ('Press_01 ') and standardize case ('press_03')
    # dict.get(key, default) takes 2 parameters:
    #   1st arg (s): the key to look up in MACHINE_ALIASES (e.g. 'P-02' -> 'PRESS_02')
    #   2nd arg (s): the fallback value returned if the key is NOT in MACHINE_ALIASES (keeps 'PRESS_01' or 'PRESS_99' as-is)
    return MACHINE_ALIASES.get(s, s)

def standardize_downtime_type(val):
    s = str(val).strip().lower()                  # Strip spaces and lowercase for case-insensitive matching
    if "prevent" in s or "pm" in s:               # Catch 'Preventive', 'preventive', 'PM', etc.
        return "Preventive"
    if "correct" in s:                            # Catch 'Corrective', 'corrective_action', etc.
        return "Corrective"
    return "Unplanned"                            # Default category for 'unplanned', 'Unplanned', etc.

# Inspect raw values BEFORE cleaning
print("Raw telemetry IDs :", sorted(df_telem["machine_id"].dropna().unique()))
print("Raw downtime types:", df_cmms["downtime_type"].value_counts().to_dict())

# TODO: apply clean_machine_code and standardize_downtime_type to df_telem, df_cmms, df_machines
#   - df_machines["machine_id"]  <- clean_machine_code
#   - df_telem["machine_id"]     <- clean_machine_code
#   - df_cmms["machine_code"]    <- clean_machine_code
#   - df_cmms["downtime_type"]   <- standardize_downtime_type


# Inspect cleaned values AFTER your .apply(...) calls
print("Clean telemetry IDs :", sorted(df_telem["machine_id"].unique()))
print("Clean downtime types:", df_cmms["downtime_type"].value_counts().to_dict())


## ⏱️ 6. Data Types, Timestamps, Deduplication & Night-Shift `S3` Rollover
### 6.1 Why Data Types Matter in Silver Parquet (`string` vs `timestamp` / `date`)
In Lab 02, `bronze_telemetry.parquet` intentionally preserved `timestamp` and `_ingestion_timestamp_utc` as raw **`string` (`object`)** text. Before parsing them into true Silver types, run three concrete experiments proving **why leaving timestamps/dates/numbers as strings in Silver breaks pipelines**:
1. **Broken Temporal Math & Methods:** `timestamp - pd.Timedelta(hours=6)` throws `TypeError`, `.dt.hour` throws `AttributeError`, and Lab 04 `.interpolate(method='time')` / Lab 05 `.resample('1h')` fail.
2. **Silent Lexicographical Ordering Traps:** String comparison compares ASCII characters left-to-right, NOT chronology (`'09/10/2026' > '01/11/2026'` is `True`; `'2026-10-01T06:00:00' > '2026-10-01 07:00:00'` is `True` because `'T' > ' '`!) — corrupting SQL filters and Parquet footer `min/max` pruning!
3. **RAM & Engine Contract:** Native `datetime64[ns]` uses `8 bytes/row` instead of Python string object pointers (**~8.5× less RAM**), and exposes native `TIMESTAMP` / `DATE` columns directly to DuckDB (`INTERVAL '2 HOURS'` in Lab 06) and BI tools!

In [ ]:
# Experiment 1: Temporal arithmetic fails on raw string timestamps
try:
    _ = df_telem["timestamp"].head(3) - pd.Timedelta(hours=6)
except TypeError as err:
    print(f"[Experiment 1] String timestamp arithmetic FAILED as expected -> TypeError: {err}")

# Experiment 2: Silent alphabetical vs chronological ordering bugs
eu_cmp_str = "09/10/2026" > "01/11/2026"
iso_cmp_str = "2026-10-01T06:00:00" > "2026-10-01 07:00:00"  # 'T' (ASCII 84) > ' ' (ASCII 32)!
num_cmp_str = "9" > "10"
print(f"[Experiment 2] String traps -> '09/10/2026' > '01/11/2026' is {eu_cmp_str}, '2026-10-01T06:00:00' > '2026-10-01 07:00:00' is {iso_cmp_str}, '9' > '10' is {num_cmp_str}!")

# TODO: Measure RAM usage of df_telem['timestamp'] as string (object) vs parsed datetime64[ns]
ram_str_bytes = None  # REPLACE WITH: df_telem["timestamp"].memory_usage(deep=True)
ram_dt_bytes = None   # REPLACE WITH: pd.to_datetime(df_telem["timestamp"], errors="coerce").memory_usage(deep=True)

if ram_str_bytes is not None and ram_dt_bytes is not None:
    print(f"[Experiment 3] RAM for 'timestamp' -> object/str: {ram_str_bytes:,} B vs datetime64[ns]: {ram_dt_bytes:,} B ({ram_str_bytes / ram_dt_bytes:.2f}x smaller!)")
else:
    print("⏳ [TODO] Compute ram_str_bytes and ram_dt_bytes above.")


### 6.2 Timestamp Parsing, Post-Normalization Deduplication & Night-Shift `S3` Rollover
1. Parse `df_telem['timestamp']` and `df_cmms['event_date']` into native **`datetime64[ns]`** (`pd.to_datetime(..., errors='coerce')`).
2. Also convert `_ingestion_timestamp_utc` into true **`datetime64[ns, UTC]`** across all 5 tables (fulfilling Lab 02's promise!).
3. Deduplicate telemetry & CMMS **after** key normalization (`keep='last'`).
4. Assign `shift_code` and **`operational_shift_date = (ts - 6h).dt.date`** — kept as native **`datetime.date`** (Parquet `DATE` / `date32[day]`, **not** `.astype(str)`!) so `00:00–05:45` maps cleanly to the shift's starting operational date.

In [ ]:
# TODO:
# 1. Parse df_telem['timestamp'] and df_cmms['event_date'] with pd.to_datetime(..., errors='coerce')
# 2. Deduplicate df_telem (on ["timestamp", "machine_id", "product_variant"]) and df_cmms (on ["ticket_id"]) with keep='last'
# 3. Convert _ingestion_timestamp_utc to UTC datetime64[ns, UTC] on all 5 DataFrames (format="ISO8601", utc=True, errors="coerce")
# 4. Assign shift_code ('S1' 06-14, 'S2' 14-22, 'S3' 22-06) and operational_shift_date = (ts - pd.Timedelta(hours=6)).dt.date (NO .astype(str)!)

if pd.api.types.is_datetime64_any_dtype(df_telem["timestamp"]):
    print("Telemetry timestamp dtype:", df_telem["timestamp"].dtype)
else:
    print("⏳ [TODO] Parse timestamps, deduplicate, and compute shift_code + operational_shift_date above.")


## 💾 7. Enforce Silver Schema Contract, Persist `silver_part1/*.parquet` & Audit Types
Before saving to Silver Part 1, enforce an explicit **Silver Schema Contract** across all 5 tables:
- Text identifier/label columns -> **`string`** (no ambiguous `object` text columns!)
- Numeric measurements & counts -> explicit **`float64`** / **`int64`**
- Operational timestamps (`timestamp`, `event_date`) -> **`timestamp[ns]`** (`INT64` in Parquet)
- Audit lineage (`_ingestion_timestamp_utc`) -> **`timestamp[ns, tz=UTC]`**
- Business date (`operational_shift_date`) -> **`date32[day]`** (`DATE`, `INT32` in Parquet)

After writing the Parquet files, compare **Bronze vs Silver Part 1 Parquet schemas**, inspect DuckDB `DESCRIBE`, and verify that Parquet footer statistics now store native **`Timestamp`** and **`Date`** bounds!

In [ ]:
# Explicit Silver Schema Contract for all 5 tables (timestamps & dates are typed separately in Section 6.2)
SILVER_SCHEMAS = {
    "telemetry": {
        "string": ["machine_id", "product_variant", "shift_code", "_source_file"],
        "float64": ["air_temp_K", "process_temp_K", "torque_Nm"],
        "int64": ["rotational_speed_rpm", "tool_wear_min", "produced_units", "scrap_units"],
    },
    "maintenance": {
        "string": ["ticket_id", "machine_code", "downtime_type", "failure_mode", "shift_code", "_source_file"],
        "int64": ["repair_duration_min"],
    },
    "machines": {
        "string": ["machine_id", "machine_name", "drive_type", "_source_file"],
        "float64": ["nominal_cycle_time_sec", "hourly_operating_cost_eur"],
    },
    "shifts": {
        "string": ["shift_id", "shift_name", "start_time", "end_time", "_source_file"],
        "int64": ["planned_duration_hours"],
    },
    "variants": {
        "string": ["variant_code", "variant_name", "material_grade", "_source_file"],
        "float64": ["unit_scrap_cost_eur"],
        "int64": ["target_hourly_output"],
    },
}

def enforce_dtypes(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    out = df.copy()  # Work on an independent copy so we don't mutate the caller's DataFrame in-place
    for col in spec.get("string", []):
        out[col] = out[col].astype("string")                                  # Pandas nullable StringDtype (no ambiguous object)
    for col in spec.get("float64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64") # Coerce invalid text -> NaN, then 64-bit float
    for col in spec.get("int64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("int64")   # Cast integer counters/IDs to 64-bit int
    return out

# 1. Apply the Silver Schema Contract to all 5 DataFrames
df_telem = enforce_dtypes(df_telem, SILVER_SCHEMAS["telemetry"])
df_cmms = enforce_dtypes(df_cmms, SILVER_SCHEMAS["maintenance"])
df_machines = enforce_dtypes(df_machines, SILVER_SCHEMAS["machines"])
df_shifts = enforce_dtypes(df_shifts, SILVER_SCHEMAS["shifts"])
df_variants = enforce_dtypes(df_variants, SILVER_SCHEMAS["variants"])

# 2. Persist all 5 typed tables to silver_part1/*.parquet (index=False avoids writing __index_level_0__)
outputs = {
    "silver_part1_telemetry.parquet": df_telem,
    "silver_part1_maintenance.parquet": df_cmms,
    "silver_machines.parquet": df_machines,
    "silver_shifts.parquet": df_shifts,
    "silver_product_variants.parquet": df_variants,
}
for fname, df_out in outputs.items():
    df_out.to_parquet(SILVER_P1 / fname, index=False, compression="snappy")

# 3. Verify string hygiene & categorical standardization before auditing schemas
assert df_telem["machine_id"].str.contains(" ").sum() == 0, "Whitespace remains in machine_id!"
assert set(df_cmms["downtime_type"]).issubset({"Unplanned", "Preventive", "Corrective"}), "Unstandardized downtime types!"

# 4. Compare Bronze vs Silver Part 1 Arrow schemas & Parquet footer min/max statistics:
#    - Arrow schema (schema_arrow): high-level logical types used in memory by Pandas/PyArrow/DuckDB
#      (e.g. string, timestamp[ns], timestamp[ns, tz=UTC], date32[day]), preserved in file metadata.
#    - Parquet physical type (physical_type): low-level on-disk binary primitive inside Column Chunks
#      (e.g. timestamp[ns] -> INT64 on disk, date32[day] -> INT32 on disk, string -> BYTE_ARRAY).
pf_bronze = pq.ParquetFile(BRONZE_STAGED / "bronze_telemetry.parquet")
pf_silver = pq.ParquetFile(SILVER_P1 / "silver_part1_telemetry.parquet")
bronze_types = {f.name: str(f.type) for f in pf_bronze.schema_arrow}
silver_types = {f.name: str(f.type) for f in pf_silver.schema_arrow}
rg_s = pf_silver.metadata.row_group(0)
silver_stats = {rg_s.column(i).path_in_schema: rg_s.column(i) for i in range(rg_s.num_columns)}

comparison = pd.DataFrame([
    {
        "column": col,
        "bronze_arrow_type": bronze_types.get(col, "<new>"),
        "silver_arrow_type": silver_types[col],
        "parquet_phys_type": silver_stats[col].physical_type,
        "footer_min": repr(silver_stats[col].statistics.min) if silver_stats[col].statistics else None,
        "footer_max": repr(silver_stats[col].statistics.max) if silver_stats[col].statistics else None,
    }
    for col in silver_types
])
print("=== BRONZE vs SILVER PART 1 PARQUET SCHEMA & FOOTER STATS ===")
print(comparison.to_string(index=False))

# 5. Assert strict Arrow/Parquet temporal & date contracts across ALL 5 saved files
for fname in outputs:
    sc = pq.ParquetFile(SILVER_P1 / fname).schema_arrow
    names = set(sc.names)
    assert "__index_level_0__" not in names, f"Index leak in {fname}!"
    ing_t = sc.field("_ingestion_timestamp_utc").type
    assert pa.types.is_timestamp(ing_t) and ing_t.tz == "UTC", f"{fname}: _ingestion_timestamp_utc must be UTC timestamp, got {ing_t}"

assert pa.types.is_timestamp(pf_silver.schema_arrow.field("timestamp").type)
assert pa.types.is_date(pf_silver.schema_arrow.field("operational_shift_date").type)
cmms_sc = pq.ParquetFile(SILVER_P1 / "silver_part1_maintenance.parquet").schema_arrow
assert pa.types.is_timestamp(cmms_sc.field("event_date").type)
assert pa.types.is_date(cmms_sc.field("operational_shift_date").type)

# 6. Verify DuckDB sees native TIMESTAMP / TIMESTAMPTZ / DATE types directly from Parquet
con = duckdb.connect()
duck_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SILVER_P1 / 'silver_part1_telemetry.parquet'}')").df()
con.close()
print("\nDuckDB DESCRIBE on silver_part1_telemetry.parquet (native TIMESTAMP / TIMESTAMPTZ / DATE!):")
print(duck_schema[["column_name", "column_type"]].to_string(index=False))

print(f"\nSaved & schema-verified all 5 Silver Part 1 Parquets in: {SILVER_P1}")
print("NOTE: PRESS_99, 2099 future dates, and -999/4850 torque values are intentionally kept here for Lab 04 Quarantine!")
